In [1]:
import numpy as np
import plotly.graph_objects as go
from terrain_class import MultiPath
from geometry import (
    generate_launch_directions,
    generate_pyramid_directions,
    batch_intersect,
    generate_surface_equal_directions,
    generate_surface_directions,
)
from scenarios import *
# Use whichever one exists in your current project.
# Option A:
from scenarios import generate_lunar_mesh_2

# Option B, if you still have the old utility:
# from utils import generate_lunar_mesh

BASE_DIR = Path.cwd().resolve().parent
tifs_dir = BASE_DIR / "Multipath" / "tifs_new"
kernels_dir = BASE_DIR / "Multipath" / "kernels"
meshes_dir = BASE_DIR / "Multipath" / "meshes"

# =============================================================================
# 2. Launch mode selector
# =============================================================================

def make_launch_dirs(
    mode,
    pos_tx,
    mesh,
    num_rays=500,
    horizon_margin_deg=3.0,
    seed=42,
):
    """
    mode options:
      - "full_sphere"
      - "lower_flat"
      - "terrain_horizon"
      - "pyramid"
      - "surface"
    """
    if mode in ("full_sphere", "lower_flat", "terrain_horizon"):
        dirs, n_eff, omega = generate_launch_directions(
            pos_tx=pos_tx,
            num_rays_full_sphere=num_rays,
            mesh=mesh,
            mode=mode,
            horizon_margin_deg=horizon_margin_deg,
        )

    elif mode == "pyramid":
        dirs, n_eff, omega = generate_pyramid_directions(
            pos_tx=pos_tx,
            num_rays=num_rays,
            mesh=mesh,
            aim_point=mesh.centroid,
            margin_frac=0.0,
        )

    elif mode == "surface":
        dirs, n_eff, omega = generate_surface_directions(
            pos_tx=pos_tx,
            num_rays=num_rays,
            mesh=mesh,
            seed=seed,
            include_centroids=True,
            use_projected_area=True,   # was the intent of area_weighted=True
            use_abs_cos=False,         # don't waste rays aiming at undersides
        )

    elif mode == "surface_equal":
        dirs, n_eff, omega = generate_surface_equal_directions(
            pos_tx=pos_tx,
            num_rays=num_rays,
            mesh=mesh,
            seed=seed,
            use_centroid_first=True,
            visible_only=False,
        )
    else:
        raise ValueError(f"Unknown launch mode: {mode!r}")

    dirs = np.asarray(dirs, dtype=np.float64)
    dirs /= np.linalg.norm(dirs, axis=1, keepdims=True)

    return dirs, n_eff, omega


# =============================================================================
# 3. Plot helper: display only the local segment near the mesh
# =============================================================================

def _ray_display_start(pos_tx, dirs, mesh, view_height):
    """
    For a far-away TX, compute where each ray enters a local display slab
    above the mesh.

    The true ray is still defined by pos_tx + t*dir.
    We only plot from z = mesh_zmax + view_height down to the hit/miss point.
    """
    pos_tx = np.asarray(pos_tx, dtype=np.float64)
    dirs = np.asarray(dirs, dtype=np.float64)

    z_top = float(mesh.bounds[1, 2] + view_height)

    starts = np.tile(pos_tx, (len(dirs), 1)).astype(np.float64)

    denom = dirs[:, 2]
    valid = np.abs(denom) > 1e-12

    t = np.zeros(len(dirs), dtype=np.float64)
    t[valid] = (z_top - pos_tx[2]) / denom[valid]

    # Only replace the start if the ray crosses the display top plane ahead
    # of the true TX. Otherwise keep the true TX.
    use_clip = valid & (t > 0.0)
    starts[use_clip] = pos_tx[None, :] + t[use_clip, None] * dirs[use_clip]

    return starts


def _ray_miss_end(pos_tx, dirs, mesh, z_ref=None):
    """
    For miss rays, draw them until they cross a reference horizontal plane.
    This lets you see whether they pass outside the mesh footprint.
    """
    pos_tx = np.asarray(pos_tx, dtype=np.float64)
    dirs = np.asarray(dirs, dtype=np.float64)

    if z_ref is None:
        z_ref = float(np.mean(mesh.vertices[:, 2]))

    denom = dirs[:, 2]
    valid = np.abs(denom) > 1e-12

    t = np.full(len(dirs), np.nan, dtype=np.float64)
    t[valid] = (z_ref - pos_tx[2]) / denom[valid]

    ends = pos_tx[None, :] + t[:, None] * dirs

    # Fallback for directions that do not cross z_ref sensibly.
    bad = (~valid) | (~np.isfinite(t)) | (t <= 0.0)
    if np.any(bad):
        span = np.linalg.norm(mesh.bounds[1] - mesh.bounds[0])
        ends[bad] = pos_tx[None, :] + dirs[bad] * span

    return ends


def _lines_xyz(starts, ends):
    """
    Convert many segments to Plotly x/y/z lists with None separators.
    """
    starts = np.asarray(starts, dtype=np.float64)
    ends = np.asarray(ends, dtype=np.float64)

    x, y, z = [], [], []

    for a, b in zip(starts, ends):
        x.extend([a[0], b[0], None])
        y.extend([a[1], b[1], None])
        z.extend([a[2], b[2], None])

    return x, y, z


def plot_launch_mode_debug(
    mesh,
    pos_tx,
    mode="pyramid",
    num_rays=500,
    max_rays_to_draw=500,
    max_dist=None,
    view_height=300.0,
    draw_misses=True,
    seed=42,
    
):
    """
    Visualize launch directions and first intersections.

    Green rays: hit the mesh.
    Red rays: miss the mesh, drawn down to the mean mesh-z plane.
    Blue points: first intersection points.
    Black marker: projected/annotated TX direction, not to scale if TX is very high.
    """
    pos_tx = np.asarray(pos_tx, dtype=np.float64)

    dirs, n_eff, omega = make_launch_dirs(
        mode=mode,
        pos_tx=pos_tx,
        mesh=mesh,
        num_rays=num_rays,
        seed=seed,
    )

    if max_dist is None:
        max_dist = float(np.linalg.norm(pos_tx - mesh.centroid) * 2.0)

    hit_mask, hit_pts, hit_faces, hit_dists = batch_intersect(
        mesh,
        np.tile(pos_tx, (len(dirs), 1)),
        dirs,
        max_dist=max_dist,
    )


    ## FIX FIX FIX
    
    n_at_hit = mesh.face_normals[hit_faces]
    backside = hit_mask & (np.einsum('ij,ij->i', dirs, n_at_hit) > 0.0)
    hit_mask[backside] = False        # underside hits are not real terrain contacts

    hit_pts = hit_pts.astype(np.float64)

    rng = np.random.default_rng(seed)
    all_ids = np.arange(len(dirs))

    if len(all_ids) > max_rays_to_draw:
        draw_ids = rng.choice(all_ids, size=max_rays_to_draw, replace=False)
    else:
        draw_ids = all_ids

    draw_hit = draw_ids[hit_mask[draw_ids]]
    draw_miss = draw_ids[~hit_mask[draw_ids]]

    starts_all = _ray_display_start(pos_tx, dirs, mesh, view_height=view_height)

    fig = go.Figure()

    # Mesh
    V = np.asarray(mesh.vertices)
    F = np.asarray(mesh.faces)

    fig.add_trace(go.Mesh3d(
        x=V[:, 0],
        y=V[:, 1],
        z=V[:, 2],
        i=F[:, 0],
        j=F[:, 1],
        k=F[:, 2],
        color="dimgray",
        opacity=0.45,
        flatshading=False,
        name="Mesh",
    ))

    # Hit rays
    if len(draw_hit) > 0:
        hx, hy, hz = _lines_xyz(
            starts_all[draw_hit],
            hit_pts[draw_hit],
        )

        fig.add_trace(go.Scatter3d(
            x=hx,
            y=hy,
            z=hz,
            mode="lines",
            line=dict(color="lime", width=3),
            opacity=0.65,
            name=f"Hit rays ({len(draw_hit)} drawn)",
        ))

        fig.add_trace(go.Scatter3d(
            x=hit_pts[draw_hit, 0],
            y=hit_pts[draw_hit, 1],
            z=hit_pts[draw_hit, 2],
            mode="markers",
            marker=dict(size=3, color="blue"),
            name="First hit points",
        ))

    # Miss rays
    if draw_misses and len(draw_miss) > 0:
        miss_end = _ray_miss_end(pos_tx, dirs[draw_miss], mesh)
        mx, my, mz = _lines_xyz(
            starts_all[draw_miss],
            miss_end,
        )

        fig.add_trace(go.Scatter3d(
            x=mx,
            y=my,
            z=mz,
            mode="lines",
            line=dict(color="red", width=2),
            opacity=0.35,
            name=f"Miss rays ({len(draw_miss)} drawn)",
        ))

    # TX annotation marker near top of local display window
    z_top = float(mesh.bounds[1, 2] + view_height)
    tx_local_marker = np.array([mesh.centroid[0], mesh.centroid[1], z_top])

    fig.add_trace(go.Scatter3d(
        x=[tx_local_marker[0]],
        y=[tx_local_marker[1]],
        z=[tx_local_marker[2]],
        mode="markers+text",
        marker=dict(size=7, color="black", symbol="diamond"),
        text=[f"TX direction<br>true z={pos_tx[2]:.1e} m"],
        textposition="top center",
        name="TX direction marker",
    ))

    hit_count = int(hit_mask.sum())
    hit_pct = 100.0 * hit_count / max(len(dirs), 1)

    title = (
        f"Launch mode: {mode} | generated={len(dirs)} | "
        f"hits={hit_count}/{len(dirs)} ({hit_pct:.2f}%)"
    )

    if np.isfinite(omega):
        title += f" | omega={omega:.3e} sr"

    fig.update_layout(
        title=title,
        template="plotly",
        scene=dict(
            xaxis_title="X [m]",
            yaxis_title="Y [m]",
            zaxis_title="Z [m]",
            aspectmode="data",
        ),
        margin=dict(l=0, r=0, b=0, t=50),
        legend=dict(
            yanchor="top",
            y=0.99,
            xanchor="left",
            x=0.01,
        ),
    )

    fig.show()

    print("=" * 70)
    print(f"mode              : {mode}")
    print(f"directions made   : {len(dirs)}")
    print(f"n_eff             : {n_eff}")
    print(f"mesh hits         : {hit_count}/{len(dirs)} = {hit_pct:.2f}%")
    print(f"drawn rays        : {len(draw_ids)}")
    print(f"drawn hit rays    : {len(draw_hit)}")
    print(f"drawn miss rays   : {len(draw_miss)}")
    print(f"TX true position  : {pos_tx}")
    print("=" * 70)

    return {
        "dirs": dirs,
        "hit_mask": hit_mask,
        "hit_pts": hit_pts,
        "hit_faces": hit_faces,
        "hit_dists": hit_dists,
    }



Class Loaded


In [3]:

# =============================================================================
# 4. Dummy mesh + examples
# =============================================================================

# Dummy lunar mesh.
# Keep this reasonably small; for launch visualization you do not need millions of triangles.
# mesh = generate_lunar_mesh_2(
#         size=1000.0,          
#         triangle_resolution=10.0,        
#         terrain_type='basin', 
#         num_craters=50, 
#         min_crater_r=3.0, 
#         max_crater_r=100.0,
#         seed=42
#     )    

# mesh = choose_mesh("florence")

t1 = MultiPath(frame = "local")
t1.load_scenario("florence")
# t1.gen_mesh(target_resolution_m=220)
mesh = t1.mesh

# If using the old function instead:
# _, _, _, _, mesh = generate_lunar_mesh(size=1000, res=80, num_craters=30, max_height=30)

# High-altitude TX in local coordinates.
# Slightly off-axis is useful because it makes pyramid behavior easier to understand.

pos_tx = np.array([0, 0, 500.0], dtype=np.float64)
num_rays = 1000
max_rays_to_draw = 1000

# Try one at a time.
out_full = plot_launch_mode_debug(
    mesh,
    pos_tx,
    mode="full_sphere",
    num_rays=num_rays,
    max_rays_to_draw=max_rays_to_draw,
    view_height=400.0,
    draw_misses=False,
)

out_lower = plot_launch_mode_debug(
    mesh,
    pos_tx,
    mode="lower_flat",
    num_rays=num_rays,
    max_rays_to_draw=max_rays_to_draw,
    view_height=400.0,
    draw_misses=False,
)

out_pyr = plot_launch_mode_debug(
    mesh,
    pos_tx,
    mode="pyramid",
    num_rays=num_rays,
    max_rays_to_draw=max_rays_to_draw,
    view_height=400.0,
    draw_misses=True,
)

out_surf = plot_launch_mode_debug(
    mesh,
    pos_tx,
    mode="surface",
    num_rays=num_rays,
    max_rays_to_draw=max_rays_to_draw,
    view_height=400.0,
    draw_misses=True,
)

out_surf = plot_launch_mode_debug(
    mesh,
    pos_tx,
    mode="surface_equal",
    num_rays=num_rays,
    max_rays_to_draw=max_rays_to_draw,
    view_height=400.0,
    draw_misses=True,
)



mode              : full_sphere
directions made   : 1000
n_eff             : 1000
mesh hits         : 176/1000 = 17.60%
drawn rays        : 1000
drawn hit rays    : 176
drawn miss rays   : 824
TX true position  : [  0.   0. 500.]
mode              : lower_flat
directions made   : 526
n_eff             : 1000
mesh hits         : 176/526 = 33.46%
drawn rays        : 526
drawn hit rays    : 176
drawn miss rays   : 350
TX true position  : [  0.   0. 500.]
mode              : pyramid
directions made   : 990
n_eff             : 990
mesh hits         : 869/990 = 87.78%
drawn rays        : 990
drawn hit rays    : 869
drawn miss rays   : 121
TX true position  : [  0.   0. 500.]
mode              : surface
directions made   : 1000
n_eff             : 1000
mesh hits         : 1000/1000 = 100.00%
drawn rays        : 1000
drawn hit rays    : 1000
drawn miss rays   : 0
TX true position  : [  0.   0. 500.]
mode              : surface_equal
directions made   : 1000
n_eff             : 1000
mesh hits  